# 00 · Configuración de la base de datos

Crea `data.db` y su esquema. Se ejecuta una sola vez al inicio del proyecto,
o cuando haga falta reconstruir alguna tabla desde cero.

| Tabla | Contenido | Columnas |
|---|---|---|
| `metadata` | Catálogo WikiArt: artista, género, movimiento | 4 |
| `mfdfa_b1` | MF-DFA, banda 6 px a 25% | 197 |
| `mfdfa_b2` | MF-DFA, banda 25% a 75% | 197 |
| `mfrenyi_b1` | MF-Rényi, banda 6 px a 25% | 183 |
| `mfrenyi_b2` | MF-Rényi, banda 25% a 75% | 183 |

Todas se ligan por `painting_id`, que es la posición de la obra en el dataset.

**La extracción de características no ocurre aquí.** Este notebook solo define
el contenedor; los notebooks 01 en adelante lo llenan.

In [ ]:
import sqlite3
import pandas as pd

import db

## Catálogo de obras

Se piden solo las columnas de etiqueta, en modo streaming, así que las
imágenes nunca se descargan.

In [ ]:
con = db.connect()
db.create_metadata_table(con)

## Tablas de características

Una tabla por método y por banda. `drop=True` reconstruye desde cero:
úsalo solo cuando quieras perder lo que haya dentro.

In [ ]:
db.create_mfdfa_b1(con, drop=True)
db.create_mfdfa_b2(con, drop=True)
db.create_mfrenyi_b1(con, drop=True)
db.create_mfrenyi_b2(con, drop=True)

## Verificación del esquema

Qué tablas existen, cuántas columnas tiene cada una y cuántas filas lleva.

In [ ]:
tablas = [f[0] for f in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
)]

for t in tablas:
    n_cols = len(list(con.execute(f'PRAGMA table_info("{t}")')))
    n_filas = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
    print(f"{t:<14} {n_cols:>4} columnas   {n_filas:>7} filas")

## Panorama del catálogo

Distribución de las etiquetas antes de aplicar cualquier filtro.

In [ ]:
pd.read_sql_query("""
    SELECT movement, COUNT(*) AS obras
    FROM metadata
    GROUP BY movement
    ORDER BY obras DESC
    LIMIT 15
""", con)

In [ ]:
pd.read_sql_query("""
    SELECT
        COUNT(*)                          AS total,
        COUNT(DISTINCT artist)            AS artistas,
        COUNT(DISTINCT movement)          AS movimientos,
        COUNT(DISTINCT genre)             AS generos
    FROM metadata
""", con)

## Categorías que superan el umbral de 200 obras

Los conteos que definen las tareas de clasificación del capítulo experimental.

Verifica cómo nombra WikiArt a las etiquetas desconocidas antes de confiar en
los filtros de abajo; la consulta siguiente las lista.

In [ ]:
pd.read_sql_query("""
    SELECT artist AS etiqueta, COUNT(*) AS obras FROM metadata
    WHERE artist LIKE '%nknown%' OR artist LIKE '%esconocid%'
    GROUP BY artist
    UNION ALL
    SELECT genre, COUNT(*) FROM metadata
    WHERE genre LIKE '%nknown%' OR genre LIKE '%esconocid%'
    GROUP BY genre
""", con)

In [ ]:
UNKNOWN_ARTIST = 'Unknown Artist'   # ajustar según la consulta anterior
UNKNOWN_GENRE   = 'Unknown Genre'

pd.read_sql_query("""
    SELECT 'movimiento' AS etiqueta, COUNT(*) AS categorias FROM (
        SELECT movement FROM metadata
        GROUP BY movement HAVING COUNT(*) >= 200
    )
    UNION ALL
    SELECT 'artista', COUNT(*) FROM (
        SELECT artist FROM metadata
        WHERE artist != :unknown_artist
        GROUP BY artist HAVING COUNT(*) >= 200
    )
    UNION ALL
    SELECT 'genero', COUNT(*) FROM (
        SELECT genre FROM metadata
        WHERE genre != :unknown_genre
        GROUP BY genre HAVING COUNT(*) >= 200
    )
""", con, params={"unknown_artist": UNKNOWN_ARTIST,
                    "unknown_genre": UNKNOWN_GENRE})

In [ ]:
con.close()